In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [ ]:
pip install openai

In [ ]:
from google.colab import files
import json

uploaded = files.upload()  # select your JSON file

with open(next(iter(uploaded)), "r", encoding="utf-8") as f:
    essays = json.load(f)


Saving essays.json to essays.json


In [ ]:
print(type(essays))
print(len(essays))
print(essays[0].keys())


<class 'list'>
147
dict_keys(['User ID', 'Essay'])


In [ ]:
SYSTEM_PROMPT = """
You are a forensic linguistics expert that reads English texts written by non-native authors to classify the native language of the author as one of:

"SPA": SPANISH
"FRE": FRENCH
"CHI": CHINESE
"GER": GERMAN
"POL": POLISH
"ARA": ARABIC
"TUR": TURKISH

Use clues such as spelling errors, word choice, syntactic patterns, and grammatical errors to decide on the native language of the author.

DO NOT USE ANY OTHER CLASS.
IMPORTANT: Do not classify any input as "ENG" (ENGLISH). English is an invalid choice.

You must provide a guess. Output two named sections: (1) "Class" with the name of the language, and (2) "Reasoning" with an explanation of your judgement with examples from the text.

You must respond with a single valid JSON object in the specified format.
"""

USER_PROMPT = """
<Input Text>
{text}

Classify the text as ONLY ONE of: SPA, FRE, CHI, GER, POL, ARA, TUR. Do not output any other class- do NOT choose "ENG" (ENGLISH). What is the closest native language of the author of this English text from the given list?

Respond in the following JSON format:

{{
  "Class": "One of SPA, FRE, CHI, GER, POL, ARA, TUR",
  "Reasoning": "An explanation of your judgement with examples from the text"
}}
"""


In [ ]:
from openai import OpenAI

client = OpenAI()

def run_nli(text, model="gpt-4.1"):
    response = client.responses.create(
        model=model,
        temperature=0.0,
        input= [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT.format(text=text)}
        ]
    )
    return response.output_text.strip()


In [ ]:
results = []
for essay in essays:
    pred_json = run_nli(essay["Essay"])
    pred_json = json.loads(pred_json)
    results.append({
        "user ID": essay["User ID"],
        "class": pred_json["Class"],
        "reasoning": pred_json["Reasoning"]
    })

In [ ]:
import json

# Save the results list to a JSON file
with open("nli_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)


In [ ]:
from google.colab import files
files.download("nli_results.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>